In [1]:
# ==========================================
# FAZ 1: KÜTÜPHANELER VE AYARLAR
# ==========================================
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import classification_report, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

# Görselleştirme ayarları
sns.set(style="whitegrid")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)

In [2]:
# ==========================================
# FAZ 2: VERİ YÜKLEME VE ÖN İŞLEME
# ==========================================
# Veri setinin yüklenmesi
flo_data = pd.read_excel('eCommerce_data_20k.xlsx')
df = flo_data.copy()

# master_id sütununu 0'dan başlayarak sıralı bir şekilde değiştirme
df["master_id"] = range(0, len(df))

# Sayısal olmayan verileri coerce ile NaN yapıp sayısal (float) tipe dönüştürme
numeric_cols = [
    "order_num_total_ever_online", "order_num_total_ever_offline",
    "customer_value_total_ever_online", "customer_value_total_ever_offline"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Toplam alışveriş ve harcamalar için Omnichannel değişkenlerin oluşturulması
df["total_order_num"] = df["order_num_total_ever_online"] + df["order_num_total_ever_offline"]
df["total_customer_value"] = df["customer_value_total_ever_online"] + df["customer_value_total_ever_offline"]

# Tarih sütunlarını datetime formatına dönüştürme
date_columns = df.columns[df.columns.str.contains("date")]
df[date_columns] = df[date_columns].apply(pd.to_datetime)

print("Veri Ön İşleme Tamamlandı. İlk 5 Satır:")
display(df[["master_id", "total_order_num", "total_customer_value"]].head())

Veri Ön İşleme Tamamlandı. İlk 5 Satır:


,master_id,total_order_num,total_customer_value
0,0,5.0,939.37
1,1,21.0,2013.55
2,2,5.0,585.32
3,3,2.0,121.97
4,4,2.0,209.98


In [3]:
# ==========================================
# FAZ 3: RFM ANALİZİ VE MÜŞTERİ SEGMENTASYONU
# ==========================================
# Analiz tarihinin belirlenmesi (maksimum tarihten 2 gün sonrası)
analysis_date = df["last_order_date"].max() + pd.Timedelta(days=2)

# RFM metriklerinin hesaplanması
rfm = df.groupby("master_id").agg({
    "last_order_date": lambda date: (analysis_date - date.max()).days,
    "total_order_num": "sum",
    "total_customer_value": "sum"
}).rename(columns={
    "last_order_date": "recency",
    "total_order_num": "frequency",
    "total_customer_value": "monetary"
})

# RFM Skorlaması (1-5 arası)
rfm["recency_score"] = pd.qcut(rfm["recency"], 5, labels=[5, 4, 3, 2, 1])
rfm["frequency_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])
rfm["monetary_score"] = pd.qcut(rfm["monetary"], 5, labels=[1, 2, 3, 4, 5])

# RF Skorunun oluşturulması ve Segment İsimlendirmeleri
rfm["RF_SCORE"] = rfm["recency_score"].astype(str) + rfm["frequency_score"].astype(str)

seg_map = {
    r'[1-2][1-2]': 'hibernating',
    r'[1-2][3-4]': 'at_Risk',
    r'[1-2]5': 'cant_loose',
    r'3[1-2]': 'about_to_sleep',
    r'33': 'need_attention',
    r'[3-4][4-5]': 'loyal_customers',
    r'41': 'promising',
    r'51': 'new_customers',
    r'[4-5][2-3]': 'potential_loyalists',
    r'5[4-5]': 'champions'
}
rfm['Segment'] = rfm['RF_SCORE'].replace(seg_map, regex=True)

# ----------------------------------------------------
# İŞ SENARYOLARI (KAMPANYA HEDEF KİTLELERİ)
# ----------------------------------------------------
rfm_final = rfm.reset_index().merge(df[["master_id", "interested_in_categories_12"]], on="master_id", how="left")

# CASE A: Yeni Kadın Ayakkabı Markası
case_a = rfm_final[(rfm_final["Segment"].isin(['champions', 'loyal_customers'])) & 
                   (rfm_final["interested_in_categories_12"].str.contains("KADIN", na=False))]
case_a["master_id"].to_csv("yeni_kadin_markasi_hedef.csv", index=False)

# CASE B: Erkek/Çocuk Ürünleri %40 İndirim
target_b = ['cant_loose', 'hibernating', 'about_to_sleep', 'new_customers']
case_b = rfm_final[(rfm_final["Segment"].isin(target_b)) & 
                   ((rfm_final["interested_in_categories_12"].str.contains("ERKEK", na=False)) | 
                    (rfm_final["interested_in_categories_12"].str.contains("COCUK", na=False)))]
case_b["master_id"].to_csv("erkek_cocuk_indirim_hedef.csv", index=False)

print("RFM Segmentleri oluşturuldu ve kampanya kitleleri CSV olarak kaydedildi.")
display(rfm[['recency', 'frequency', 'monetary', 'Segment']].head())

RFM Segmentleri oluşturuldu ve kampanya kitleleri CSV olarak kaydedildi.


,recency,frequency,monetary,Segment
master_id,,,,
0,95,5.0,939.37,loyal_customers
1,105,21.0,2013.55,loyal_customers
2,186,5.0,585.32,at_Risk
3,135,2.0,121.97,about_to_sleep
4,86,2.0,209.98,about_to_sleep


In [4]:
# ==========================================
# FAZ 4: MAKİNE ÖĞRENMESİ (CHURN & CLTV)
# ==========================================

# 1. K-MEANS KÜMELEME (Sadece RFM metrikleri ile)
rfm_k = rfm[['recency', 'frequency', 'monetary']]
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_k)

kmeans = KMeans(n_clusters=4, init='k-means++', max_iter=300, n_init=10, random_state=42)
rfm['KMeans_Clusters'] = kmeans.fit_predict(rfm_scaled)

# Ana veri ile RFM tablosunu makine öğrenmesi ve dışa aktarım için BİRLEŞTİRME
merged_df = pd.merge(rfm.reset_index(), df, on='master_id', how='inner')

# ----------------------------------------------------
# 2. LOGISTIC REGRESSION İLE CHURN TAHMİNİ
# ----------------------------------------------------
# Hedef değişken: 90 günden fazla süredir gelmeyenleri Churn (1) kabul ediyoruz
merged_df['is_churn'] = (merged_df['recency'] > 90).astype(int)

X_churn = merged_df[['frequency', 'monetary', 'KMeans_Clusters']]
y_churn = merged_df['is_churn']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_churn, y_churn, test_size=0.2, random_state=42)

log_model = LogisticRegression(class_weight='balanced', random_state=42)
log_model.fit(X_train_c, y_train_c)
y_pred_c = log_model.predict(X_test_c)

print("--- LOGISTIC REGRESSION CHURN MODEL SONUÇLARI ---")
print(classification_report(y_test_c, y_pred_c))

# ----------------------------------------------------
# 3. RANDOM FOREST İLE CLTV TAHMİNİ
# ----------------------------------------------------
# ML modeli için kural tabanlı hedef (Target) oluşturma
lifespan_months = 24 
merged_df['AOV'] = merged_df['monetary'] / merged_df['frequency']
merged_df['CV'] = merged_df['AOV'] * merged_df['frequency'] 
merged_df['Target_CLTV'] = merged_df['CV'] * lifespan_months

X_cltv = merged_df[['order_num_total_ever_online', 'order_num_total_ever_offline', 
                    'customer_value_total_ever_online', 'customer_value_total_ever_offline']]
y_cltv = merged_df['Target_CLTV']

X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(X_cltv, y_cltv, test_size=0.2, random_state=42)

rf_reg_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg_model.fit(X_train_v, y_train_v)
y_pred_v = rf_reg_model.predict(X_test_v)

print("\n--- RANDOM FOREST CLTV TAHMİN MODELİ SONUÇLARI ---")
print(f"R2 Skoru (Açıklanabilirlik): {r2_score(y_test_v, y_pred_v):.2f}")
print(f"RMSE (Hata Payı): {np.sqrt(mean_squared_error(y_test_v, y_pred_v)):.2f}")

--- LOGISTIC REGRESSION CHURN MODEL SONUÇLARI ---
              precision    recall  f1-score   support

           0       0.65      0.93      0.77      1670
           1       0.93      0.64      0.76      2319

    accuracy                           0.77      3989
   macro avg       0.79      0.79      0.77      3989
weighted avg       0.82      0.77      0.76      3989


--- RANDOM FOREST CLTV TAHMİN MODELİ SONUÇLARI ---
R2 Skoru (Açıklanabilirlik): 0.99
RMSE (Hata Payı): 1143.88


In [ ]:
# ==========================================
# FAZ 5: POWER BI DASAHOARD İÇİN VERİ AKTARIMI
# ==========================================

# Modellerin ürettiği sonuçları tüm veri için tahmin edip tabloya ekleme
merged_df['Churn_Probability'] = log_model.predict_proba(X_churn)[:, 1] 
merged_df['Predicted_Churn'] = log_model.predict(X_churn)
merged_df['Predicted_CLTV'] = rf_reg_model.predict(X_cltv)

# Sadece Power BI'da raporlanacak öz ve temiz sütunları seçme
export_columns = [
    'master_id', 'order_channel', 'last_order_channel', 
    'interested_in_categories_12', 'Segment', 'KMeans_Clusters', 
    'recency', 'frequency', 'monetary', 
    'Predicted_Churn', 'Churn_Probability', 'Predicted_CLTV'
]

power_bi_df = merged_df[export_columns].copy()

# Dosyayı kaydetme
power_bi_df.to_csv("FLO_PowerBI_Dataset.csv", index=False)
print("Power BI için temizlenmiş veri seti başarıyla oluşturuldu: 'FLO_PowerBI_Dataset.csv'")
display(power_bi_df.head())